# Graph edge types

Topology-only and fusion models use the full graph, no-DDI graph, or DDI-only graph on the same 25 folds.

The saved tables can be viewed without rerunning the models.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../../data/README.md) and
[DataPreparation.ipynb](../../../data/DataPreparation.ipynb), you can rerun the
ablation and regenerate the tables using the functions in
[`src/dtpkg/ablation/`](../../../src/dtpkg/ablation/).

## Paths

Fill in `PROJECT_ROOT` to read the saved tables. The remaining placeholders are
needed for training; `REFERENCE_DIR` is the completed [transductive run](../../04_transductive_fusion/README.md).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout
DATA_DIR = ...  # Prepared private data
GRAPH_PATH = ...  # Canonical GraphML network
REFERENCE_DIR = ...  # Completed transductive run from experiment 04

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

EXPERIMENT = PROJECT_ROOT / "experiments" / "06_ablation" / "graph_edge_types"
TABLES = EXPERIMENT / "tables"
RESULTS_ROOT = EXPERIMENT / "results" / "runs"
OUTPUT_DIR = EXPERIMENT / "results" / "table_exports"
RUN_TRAINING = False
STUDY = "edge_types"

## Saved results

The tables use fold means and sample SD. Paired differences use approximate corrected-CV intervals.

In [ ]:
from IPython.display import display
from dtpkg.ablation.public_tables import load_report, performance_table, contrast_table

report = load_report(TABLES, study=STUDY)
display(performance_table(report))
display(contrast_table(report))

,Model,Graph,AUROC,F1
0,Latent gated fusion,Full,0.9885 ± 0.0033,0.9528 ± 0.0079
1,Latent gated fusion,No DDI,0.9893 ± 0.0031,0.9545 ± 0.0094
2,Latent gated fusion,DDI only,0.9881 ± 0.0040,0.9521 ± 0.0088
3,Topology only,Full,0.9450 ± 0.0059,0.8674 ± 0.0094
4,Topology only,No DDI,0.8744 ± 0.0135,0.7722 ± 0.0203
5,Topology only,DDI only,0.9377 ± 0.0093,0.8657 ± 0.0113
6,MeSH only (original),—,0.9666 ± 0.0047,0.9063 ± 0.0080


,Model,Contrast,ΔAUROC,ΔF1
0,Latent gated fusion,Full − no DDI,-0.00086,-0.00165
1,Latent gated fusion,Full − DDI only,+0.00035,+0.00074
2,Topology only,Full − no DDI,+0.07065**,+0.09515**
3,Topology only,Full − DDI only,+0.00738,+0.00170
4,Reference,No-DDI fusion − MeSH only (original),+0.02274**,+0.04821**


No-DDI retains every non-DDI relation. Validation and test positive DDIs
are masked before extracting features. The five AUROC/F1 contrasts form a
10-test Holm family. No-DDI fusion versus MeSH-only compares complete model pipelines.

## Regenerate the results

Complete the [transductive experiment](../../04_transductive_fusion/TransductiveFusion.ipynb)
and set `REFERENCE_DIR` to its run directory, then enable `RUN_TRAINING`.
The run reuses compatible transductive fits and trains the remaining arms.

In [ ]:
if RUN_TRAINING:
    from dtpkg.ablation.workflow import prepare_study
    from dtpkg.ablation.public_tables import export_tables

    study = prepare_study(
        study=STUDY,
        data_dir=DATA_DIR,
        graph_path=GRAPH_PATH,
        reference_dir=REFERENCE_DIR,
        results_root=RESULTS_ROOT,
    )
    study.run()
    export_tables(study.output_dir, OUTPUT_DIR, study=STUDY)
    new_report = load_report(OUTPUT_DIR, study=STUDY)
    display(performance_table(new_report))
    display(contrast_table(new_report))